# Dynamic portfolio allocation with tabular Q-learning — Python engine parity edition

This notebook imports **the same `asset_agent.py` implementation executed in the website through Pyodide**. It uses the identical data, default parameters, seeded generator, training transitions, reward, Q-table, and evaluation routines.

Run from the repository root or the `notebooks/` folder. The repeated 2018–19 historical weeks are an instructional example, not independent training episodes.

## 1. Specify the MDP, before writing code

| MDP component | This example |
|---|---|
| State | Sign of past **4-week momentum** (up/down); sign of past **12-week momentum** (up/down); **previous target weight** (0%, 50%, 100%) |
| Action | New weekly target weight in Apple: 0%, 50%, 100%; remainder held as non-interest-bearing cash |
| Transition | Historical next-week Apple return (exogenous to a price-taking investor), and change in previous target weight |
| Reward | Weekly net return **minus an optional downside penalty** (defined below) |
| Policy | One allocation for each of the 12 observable states |
| Learning | Tabular Q-learning with ε-greedy exploration; Bellman sample updates |

The state has **2 × 2 × 3 = 12 combinations**, and 3 actions, giving **36 Q-values**. This is larger than the original one-signal, six-state notebook, but remains interpretable.

The reward in **decimal return units** is

\[r_{t+1}= w_t R_{t+1}-c\lvert w_t-w_{t-1}\rvert-\lambda\max(-w_tR_{t+1},0).\]

Here, \(c\) is the proportional cost of changing the target weight and \(\lambda\) is an *illustrative* downside penalty. The downside penalty changes the RL objective; it is **not** a trading cost, and is never deducted when reporting realized portfolio wealth. The evaluation reports actual net returns separately.

\[Q(s,a)\leftarrow Q(s,a)+\alpha\big[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\big].\]

The financial state is only an **approximate** Markov description: past two momentum signs do not fully characterize market dynamics. This is an educational abstraction, not a validated economic model.

**Running in Google Colab?** Run the next cell first. It loads the agent code and data. Locally it does nothing.

In [ ]:
# Colab setup: downloads the workshop code and data. Does nothing outside Colab.
import sys, os
if "google.colab" in sys.modules:
    if not os.path.exists("/content/rl-workshop"):
        !git clone -q --depth 1 https://github.com/jakob284/rl-workshop.git /content/rl-workshop
    %cd /content/rl-workshop/notebooks

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
root = next((p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent] if (p / "asset_agent.py").exists()), None)
assert root is not None, "Keep asset_agent.py next to the site, or run from the repository root"
sys.path.insert(0,str(root))
from asset_agent import PortfolioAgent, DEFAULTS, greedy
df = pd.read_csv(root / "data/historical_normalized_weekly_stocks_2018_2019.csv",parse_dates=["date"])
prices = df["AAPL"].to_numpy(dtype=float).round(8)
dates = df["date"].dt.strftime("%Y-%m-%d").tolist()
print(f"Dataset: {len(prices)} observations, {dates[0]} through {dates[-1]}")


## 2. Create point-in-time signals and chronological train/test split

At decision time \(t\), the state uses only prices **up to and including** \(t\). The agent then takes an action, earns the return from \(t\) to \(t+1\), and observes the next state. **No random train/test shuffling**.

Training ends at the first evaluation date: that price is observed at the boundary, but *no return after that boundary* is used in training. Evaluated rewards start with the next, genuinely unseen return.

In [ ]:
engine=PortfolioAgent(prices,dates,**DEFAULTS)
train_t,test_t=engine.train_t,engine.test_t
short_up,long_up=engine.up4,engine.up12
WEIGHTS=np.array([0,0.5,1.0])
def state_id(t,previous): return engine.state_id(t,previous)
print(f"Training weeks {len(train_t)}; evaluation weeks {len(test_t)}; 12 states / 36 action values")


## 3. Train the Q-learning policy

We cycle over the **same** historical training dates for multiple episodes. The Python and browser versions use the same seeded pseudorandom generator for consistent examples. Replays provide additional optimization passes, **not** new independent observations. An ε-greedy policy explores; the evaluation policy is frozen and greedy.

In [ ]:
# EXACT same function calls and parameter values as the static web demo
engine.step_many(len(engine.train_t) * engine.budget)
Q = np.asarray(engine.q)
train_visits=np.asarray(engine.visits)
print(f"Training passes: {engine.passes}, last greedy training reward {engine.greedy_history[-1]:.4f}")
print("Python Q-table shape:",Q.shape)


## 4. Show what the agent learned

A Q-table is interpretable: for each combination of short momentum, long momentum and previous allocation, identify the target weight with the highest estimated action value. **Unvisited states should not be interpreted as learned expertise.**

In [ ]:
rows=[]
for m4 in [False,True]:
    for m12 in [False,True]:
        for previous in range(3):
            s=(2*int(m4)+int(m12))*3+previous
            rows.append({"4w":"Up" if m4 else "Down","12w":"Up" if m12 else "Down","previous":WEIGHTS[previous],"Q(0%)":Q[s,0],"Q(50%)":Q[s,1],"Q(100%)":Q[s,2],"chosen":WEIGHTS[greedy(Q[s])],"visits":train_visits[s]})
display(pd.DataFrame(rows).round(4))


## 5. Freeze the policy and compare against honest benchmarks

All strategies start the evaluation period from cash and face the **same** transaction cost. Benchmarks include 100% Apple, 50% Apple rebalanced weekly, 100% cash, and a simple two-horizon momentum rule.

**Evaluation net return** subtracts *actual modeled trading costs*, **not** the training-only downside penalty. Annualization (52 weeks) on this very short test window is unstable and should not be used for investment claims.

In [ ]:
strategies = engine.evaluate()
summary = pd.DataFrame([{"Strategy":v["name"],"Net total return":v["totalReturn"],"Max drawdown":v["maxDrawdown"],"Total turnover":v["turnover"],"Allocation changes":v["changes"]} for v in strategies])
display(summary.round(4))


In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
for result in strategies:
    ax.plot(np.arange(len(result['wealth'])), result['wealth'], label=result['name'])
ax.set(title='Illustrative evaluation-period portfolio wealth · starts at 1.00',
       xlabel='Evaluation weeks elapsed', ylabel='Net wealth index')
ax.grid(alpha=.2); ax.legend(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10,3))
ax.step(test_t, [entry['weight'] for entry in strategies[0]['outcome']], where='post')
ax.set(title='RL agent allocation in test period', xlabel='Decision time (row index)',
       ylabel='Apple weight', ylim=(-.05,1.05))
ax.grid(alpha=.2); plt.tight_layout(); plt.show()

## 6. Workshop discussion & limitations

1. **What is learned?** Inspect Q-values and state visits. Does the learned policy change with the two signals, or mostly choose one allocation?
2. **Do the extra features help?** More states can improve expressiveness but make state-action coverage worse when only ~60 training decisions are available.
3. **What do costs change?** Try 0, 10, and 50 bps and compare turnover and net outcomes. Re-train for each cost; changing only evaluation cost would mismatch the objective.
4. **What does downside penalty mean?** Change `DOWNSIDE_PENALTY` and re-run. It is a *utility preference*, not a cash deduction; better training reward does not guarantee higher realized return.
5. **Why not PPO yet?** 105 weekly observations do not justify a high-capacity neural policy, and the tabular agent is easy to audit.
6. **What should a serious follow-up add?** Longer point-in-time total-return data, multiple assets, a credible simulator, cash interest, portfolio constraints, proper mark-to-market weights, walk-forward validation, robust non-RL optimizers, risk metrics, and tests across independent market regimes.

**Methodological warning:** The observed return sequence is not affected by our trades, the state is not rigorously Markov, and model selection on the test period would destroy the holdout. Repeated training over a short static history can encourage memorization. This is *not a demonstration of economic superiority over classical stochastic control*.

**Sources:** The price CSV is derived from Plotly's public sample `plotly.express.data.stocks()` (weekly normalized 2018–2019). Scherer (2020), “Maschinelles Lernen im Asset Management,” *Absolut|report*, pp. 46–51, motivates examining dynamic rebalancing with signals and costs but does not establish RL outperformance.

## 7. Parameter sensitivity (real model reruns)

Each configuration creates a **new Python agent** with zero-initialized Q-values, trains on the same 64 historical weeks with the same seed, and evaluates the frozen policy on the same 28 later weeks. Reusing this evaluation set for model selection would compromise its holdout status. These comparisons illustrate sensitivity, not statistical generalization.


In [ ]:
experiments={
    "Baseline":{},
    "No costs / no penalty":{"cost_bps":0,"risk":0},
    "High cost / penalty":{"cost_bps":50,"risk":2},
    "Low discount":{"gamma":0.1}
}
comparison=[]
for name,changes in experiments.items():
    config={**DEFAULTS,**changes}
    agent=PortfolioAgent(prices,dates,**config)
    agent.step_many(len(agent.train_t)*agent.budget)
    metric=agent.evaluate()[0]
    comparison.append({"Configuration":name,"Net test return":metric["totalReturn"],"Test changes":metric["changes"],"Final greedy training reward":agent.greedy_history[-1]})
display(pd.DataFrame(comparison).round(4))
